# Kani Advanced

Kani's own transformer, made bigger and pretrained **from scratch** on Kaggle's free GPUs, with no pretrained weights.

- **Same architecture and tokenizer as Kani** (rotary attention, tied embeddings, Kani's 4,096-token BPE), so the
  result loads straight into Kani's terminal app and VS Code extension.
- **About 205M parameters** (big Kani v1 has 88M): width 1024, 16 layers, 1,024-token context.
- **Better text:** FineWeb-Edu (educational web pages) and Cosmopedia v2 (textbook-style writing), both public.
  Each session downloads new text; the goal is 4 billion tokens (big Kani v1 saw 164 million).
- **What to expect:** roughly GPT-2-class writing and general knowledge after the full run. That is a big step
  up for Kani, and still far from GPT-5, which is trained on vastly more text with far more compute.

## Setup (once)

1. **Settings → Accelerator → GPU T4 x2.**
2. **Settings → Internet → On** (Session options in the right-hand panel). The text is downloaded from Hugging Face.
3. **Save Version → Save & Run All (Commit).** About 10 minutes of downloading and tokenizing, then about 10 hours of
   training. You can close the tab.

## Every next session

**Add Input → Your Work → Notebooks →** this notebook (its latest version), then **Save & Run All** again. It
continues from `kani-advanced/checkpoint.pt` on new text. About 4–5 sessions reach the 4-billion-token goal.

`continue_from_previous` is True, so a session with that input missing stops within a minute with an error
instead of quietly training from zero. For the very first session of a new run, set it to False.

## Output (the version's Output tab)

- `kani-advanced/kani-advanced.npz`: the model, in Kani's checkpoint format
- `kani-advanced/samples.json`: what it writes after a few prompts, to see it improve
- `kani-advanced/checkpoint.pt` and `progress.json`: for continuing

In [ ]:
# Model size. Keep these the same in every session of one run.
#   width 768,  layers 12 ->  88M parameters (big Kani v1's size)
#   width 1024, layers 16 -> 205M (default)
#   width 1024, layers 24 -> 306M (largest Kani supports; needs ~1.5x the sessions)
SETTINGS = {
    'width': 1024,
    'layers': 16,
    'heads': 16,
    'context': 1024,
    'target_tokens': 4_000_000_000,  # the whole run; the learning rate winds down over the last 20%
    'hours': 10.0,                   # training time per session (Kaggle stops sessions at 12 hours)
    'micro_batch': 4,                # sequences per GPU at once; lower it if you see "CUDA out of memory"
    # True after the first session: stop at once (instead of training from zero for 10 hours)
    # when this notebook's earlier output is not attached as an input.
    'continue_from_previous': True,
}

import json
json.dump(SETTINGS, open('settings.json', 'w'), indent=1)
print(json.dumps(SETTINGS, indent=1))

In [ ]:
import torch, urllib.request
gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print('PyTorch', torch.__version__, '| GPUs:', gpus or 'none: set Settings → Accelerator → GPU T4 x2')
try:
    urllib.request.urlopen('https://huggingface.co', timeout=15)
    print('Internet: on')
except OSError as error:
    raise SystemExit(f'Cannot reach huggingface.co ({error}). Is Settings → Internet on (Session options panel)?')

Kani's tokenizer (the same file as `data/big-v1/tokenizer.json` in the KelusModelB1 repository) and the training script (`kaggle/kani_advanced.py`).

In [ ]:
%%writefile kani_tokenizer.json
{"kind":"bpe","merges":[[32,32],[261,261],[116,104],[114,101],[105,110],[101,114],[111,110],[32,97],[10,262],[101,110],[32,263],[10,10],[97,116],[111,114],[108,101],[115,116],[271,101],[115,101],[32,99],[100,101],[270,116],[97,108],[262,262],[114,111],[105,267],[32,105],[32,116],[97,110],[99,116],[265,103],[32,102],[32,111],[109,101],[97,114],[99,101],[32,115],[32,98],[101,100],[32,119],[32,112],[105,116],[32,265],[109,112],[10,261],[100,111],[268,110],[32,264],[115,115],[117,114],[112,101],[269,262],[32,101],[117,116],[287,111],[117,110],[109,281],[104,101],[286,115],[105,99],[32,110],[108,111],[96,96],[101,115],[292,102],[99,111],[103,101],[108,97],[97,100],[32,109],[116,266],[105,115],[114,97],[114,105],[99,104],[264,102],[105,108],[269,283],[306,100],[105,102],[117,101],[273,285],[32,83],[117,108],[32,118],[108,105],[112,116],[111,100],[111,116],[101,98],[261,32],[97,115],[101,120],[118,101],[297,101],[279,111],[82,101],[278,108],[116,309],[357,102],[84,317],[358,110],[120,335],[267,116],[101,289],[32,276],[97,293],[116,121],[273,101],[304,32],[108,121],[32,117],[32,280],[269,261],[65,80],[291,274],[115,266],[87,349],[275,316],[374,73],[32,104],[118,281],[322,96],[100,105],[300,284],[112,266],[98,275],[99,267],[274,116],[305,99],[48,48],[106,364],[105,263],[305,109],[115,111],[393,362],[99,107],[303,275],[80,121],[312,120],[98,391],[282,340],[112,112],[289,285],[264,110],[271,273],[97,326],[111,108],[105,103],[73,78],[32,48],[32,293],[32,274],[73,110],[116,116],[352,116],[76,69],[266,115],[286,116],[32,278],[97,397],[105,275],[32,100],[272,350],[284,119],[32,267],[102,102],[268,108],[283,283],[310,99],[299,104],[32,65],[117,316],[101,109],[299,392],[97,386],[343,116],[280,102],[320,348],[114,274],[32,67],[262,32],[268,115],[326,116],[327,308],[263,347],[107,101],[297,121],[112,264],[105,100],[101,108],[288,100],[102,274],[346,285],[69,82],[116,114],[85,83],[72,84],[108,265],[284,109],[10,269],[111,117],[117,264],[429,339],[106,115],[32,49],[101,119],[121,310],[32,68],[101,116],[118,266],[32,84],[32,103],[292,400],[100,266],[105,122],[303,273],[268,264],[278,100],[105,276],[32,69],[344,401],[32,345],[296,117],[117,112],[341,115],[117,109],[118,105],[288,295],[333,98],[389,115],[97,112],[105,353],[111,280],[113,117],[271,331],[102,97],[65,76],[315,403],[267,101],[105,293],[121,110],[424,376],[105,109],[279,288],[32,398],[385,367],[102,101],[112,97],[321,119],[387,276],[104,116],[265,116],[273,298],[411,445],[65,84],[97,278],[116,111],[264,361],[279,363],[365,114],[458,101],[32,66],[109,111],[282,108],[273,97],[32,121],[97,121],[97,265],[104,331],[367,310],[79,78],[111,119],[307,361],[288,115],[112,284],[114,439],[325,280],[463,319],[99,444],[294,116],[263,267],[278,116],[286,102],[97,118],[32,51],[69,536],[301,121],[117,275],[313,101],[32,73],[32,78],[99,264],[279,104],[312,378],[116,101],[295,308],[272,10],[336,108],[507,404],[291,459],[475,101],[305,119],[452,109],[315,100],[32,76],[32,360],[117,426],[115,104],[263,266],[114,301],[279,267],[389,432],[32,70],[111,264],[280,100],[99,333],[302,330],[337,262],[112,313],[116,97],[270,100],[384,506],[97,109],[312,381],[382,464],[538,485],[32,80],[69,381],[264,328],[347,548],[32,85],[275,289],[319,104],[32,359],[301,285],[414,489],[105,114],[315,99],[520,290],[69,378],[291,498],[105,323],[288,103],[32,79],[84,454],[97,396],[101,101],[110,366],[268,114],[309,295],[325,108],[111,112],[286,303],[276,488],[296,104],[296,111],[446,121],[559,295],[32,416],[115,290],[117,103],[32,108],[105,278],[371,278],[105,280],[32,315],[105,98],[111,115],[576,346],[32,87],[97,289],[102,598],[115,117],[273,274],[430,593],[320,366],[67,111],[83,116],[404,116],[417,285],[67,363],[97,99],[97,263],[276,114],[480,120],[32,379],[264,100],[282,117],[299,558],[482,276],[32,77],[108,108],[270,266],[308,285],[457,77],[83,83],[111,313],[32,114],[32,383],[114,534],[288,116],[301,275],[419,116],[84,529],[101,99],[111,107],[356,361],[486,98],[505,541],[320,466],[399,420],[84,605],[294,121],[567,266],[619,672],[32,305],[82,69],[86,71],[268,596],[526,461],[116,662],[276,433],[282,370],[329,590],[336,547],[10,373],[76,479],[121,682],[287,467],[82,73],[328,266],[496,295],[84,69],[113,340],[114,121],[321,103],[343,100],[103,332],[105,118],[117,276],[355,280],[32,572],[265,562],[101,303],[264,115],[430,270],[668,266],[101,381],[328,100],[394,275],[105,112],[264,583],[109,98],[117,115],[288,326],[291,421],[420,115],[648,323],[294,100],[307,115],[332,121],[371,478],[83,678],[103,266],[414,112],[427,394],[97,353],[109,347],[268,402],[69,120],[111,697],[627,685],[32,317],[105,97],[112,327],[274,107],[111,99],[337,283],[380,351],[515,409],[84,72],[356,618],[402,388],[32,321],[65,82],[100,290],[284,484],[294,103],[309,639],[332,293],[344,640],[496,436],[544,97],[32,377],[71,76],[97,334],[112,406],[115,99],[577,692],[32,50],[109,105],[266,439],[409,71],[112,105],[32,72],[109,273],[448,115],[32,86],[79,77],[387,710],[655,76],[109,109],[274,121],[356,102],[679,549],[32,275],[82,65],[83,629],[263,101],[295,453],[645,717],[307,328],[380,727],[428,428],[767,281],[73,531],[99,308],[372,752],[464,362],[73,102],[100,525],[104,725],[407,509],[551,499],[109,108],[109,301],[112,104],[704,274],[32,270],[32,443],[74,753],[84,80],[97,301],[99,108],[275,100],[284,564],[286,100],[637,476],[694,276],[32,413],[102,265],[111,111],[476,732],[501,99],[609,103],[115,108],[116,415],[291,793],[295,115],[565,740],[613,388],[73,71],[98,502],[394,610],[449,263],[457,802],[519,528],[10,311],[105,120],[268,99],[270,115],[279,444],[295,346],[321,396],[355,813],[801,778],[79,400],[99,363],[110,348],[300,111],[307,785],[377,755],[798,282],[85,110],[299,349],[325,476],[342,586],[344,718],[112,121],[282,278],[597,276],[299,351],[320,111],[330,115],[343,108],[427,108],[32,281],[69,110],[78,68],[97,98],[102,421],[455,121],[501,580],[533,115],[570,266],[611,274],[863,120],[105,111],[119,803],[274,100],[50,53],[84,500],[287,500],[297,313],[355,303],[422,525],[483,98],[497,76],[522,502],[83,101],[83,586],[268,116],[273,290],[279,524],[408,511],[97,107],[97,402],[98,111],[105,101],[108,319],[325,474],[355,108],[387,100],[491,105],[65,110],[101,334],[114,340],[330,110],[371,620],[483,743],[32,327],[50,48],[108,266],[268,764],[280,120],[471,529],[592,274],[792,115],[117,289],[118,298],[264,289],[292,569],[342,607],[371,112],[408,110],[49,48],[49,50],[73,67],[84,497],[265,614],[268,100],[279,749],[336,100],[425,101],[552,270],[816,621],[84,467],[84,843],[292,453],[300,97],[320,101],[383,111],[615,731],[79,82],[84,660],[98,606],[105,303],[112,294],[287,415],[293,263],[311,32],[383,118],[80,327],[105,282],[117,932],[85,66],[115,625],[116,332],[276,273],[279,282],[321,98],[633,98],[687,653],[78,499],[98,376],[105,327],[111,453],[116,323],[118,401],[265,298],[299,101],[300,294],[379,774],[425,370],[568,388],[603,298],[641,415],[818,290],[70,274],[77,878],[102,459],[103,511],[120,426],[306,121],[345,98],[508,263],[720,436],[815,432],[82,76],[111,309],[279,264],[296,463],[297,502],[342,970],[698,109],[943,265],[73,741],[118,282],[273,492],[294,631],[310,289],[345,276],[511,795],[630,987],[32,332],[68,101],[70,70],[101,294],[102,498],[266,121],[273,462],[284,108],[316,282],[380,451],[468,79],[554,115],[666,889],[98,674],[112,267],[117,278],[315,116],[319,493],[320,707],[329,527],[371,376],[67,104],[67,656],[99,97],[116,602],[117,98],[118,351],[121,115],[264,101],[284,652],[291,851],[292,470],[302,614],[550,102],[97,103],[98,121],[98,450],[262,261],[294,105],[385,1019],[32,82],[265,521],[292,313],[310,896],[422,331],[544,408],[638,712],[709,588],[956,386],[32,762],[68,769],[78,79],[85,76],[272,261],[276,540],[296,688],[299,317],[1042,953],[32,284],[32,667],[73,116],[80,284],[99,513],[105,298],[112,114],[114,267],[116,267],[118,97],[267,370],[273,323],[287,660],[319,282],[351,107],[353,108],[473,115],[761,654],[32,617],[49,51],[81,85],[87,865],[99,282],[121,713],[271,270],[273,455],[487,295],[676,323],[1076,407],[32,71],[68,69],[69,83],[119,104],[265,330],[293,445],[294,334],[296,366],[297,111],[300,264],[300,540],[307,809],[319,115],[327,103],[384,487],[408,104],[450,108],[461,115],[482,446],[523,353],[32,53],[102,116],[103,110],[120,65],[273,334],[274,474],[313,323],[325,303],[329,97],[334,966],[342,867],[768,401],[70,421],[79,88],[101,1001],[116,298],[265,579],[301,115],[321,328],[365,368],[535,506],[628,595],[642,317],[758,489],[777,87],[951,282],[111,109],[116,435],[468,1126],[523,910],[763,83],[1062,1043],[1130,1132],[105,466],[111,278],[115,561],[119,392],[119,825],[267,103],[321,116],[329,574],[333,120],[336,330],[384,1034],[418,115],[474,290],[494,105],[787,362],[75,101],[97,264],[109,115],[114,276],[269,32],[294,443],[308,406],[354,270],[448,1057],[508,295],[527,115],[955,470],[78,348],[100,368],[268,308],[302,116],[365,540],[830,556],[840,512],[937,388],[69,68],[82,270],[98,294],[99,524],[101,328],[267,263],[271,288],[276,332],[288,1021],[302,276],[333,323],[351,814],[366,854],[383,289],[427,509],[437,960],[472,747],[700,1128],[990,266],[32,120],[66,121],[102,330],[226,128],[268,98],[296,561],[352,833],[372,811],[379,919],[418,433],[431,859],[455,534],[609,721],[644,909],[946,416],[32,89],[65,114],[69,84],[78,366],[79,110],[105,386],[120,97],[269,350],[277,121],[278,115],[279,516],[291,97],[300,266],[302,517],[317,396],[382,993],[440,1197],[469,334],[494,1000],[735,295],[822,78],[880,907],[1005,1203],[54,52],[77,69],[83,111],[98,516],[116,100],[342,1224],[399,780],[422,339],[450,278],[472,699],[483,334],[484,275],[503,282],[1150,121],[83,84],[83,455],[83,688],[99,115],[101,276],[109,406],[110,290],[112,120],[116,348],[291,750],[292,385],[301,266],[302,579],[327,1031],[334,274],[355,772],[451,719],[48,49],[97,303],[101,112],[103,806],[116,500],[119,349],[274,103],[274,109],[302,555],[312,756],[313,462],[317,469],[325,109],[356,328],[448,654],[563,341],[613,378],[641,281],[658,315],[735,121],[32,106],[67,84],[97,346],[100,319],[102,343],[108,368],[110,407],[264,886],[268,1191],[275,516],[291,1116],[313,111],[325,746],[329,111],[351,267],[384,556],[514,115],[552,1003],[591,110],[644,290],[857,695],[1147,967],[1171,1299],[1249,467],[1284,512],[32,771],[80,266],[80,294],[83,469],[109,97],[294,734],[329,493],[329,700],[332,623],[344,1033],[345,281],[353,321],[390,55],[515,786],[533,298],[535,1301],[584,115],[737,432],[751,1111],[1222,1225],[32,52],[32,521],[70,84],[103,479],[103,747],[110,491],[122,270],[284,1329],[294,107],[298,734],[354,290],[426,542],[898,282],[950,805],[32,836],[69,78],[70,850],[73,84],[87,736],[110,266],[110,466],[264,308],[267,281],[307,108],[311,261],[312,494],[325,310],[330,109],[364,285],[390,52],[394,108],[407,319],[416,378],[419,101],[472,653],[501,263],[587,294],[638,368],[655,1355],[665,313],[804,117],[831,285],[841,290],[892,274],[1096,280],[49,49],[76,111],[87,703],[99,274],[104,97],[105,450],[108,290],[111,564],[119,703],[275,120],[291,265],[297,834],[299,570],[313,1061],[345,972],[352,99],[352,420],[440,656],[517,611],[519,281],[921,100],[982,1058],[50,50],[65,945],[73,76],[79,80],[79,84],[99,309],[99,1124],[108,273],[112,111],[265,101],[268,289],[270,295],[280,537],[300,406],[300,643],[306,503],[307,1148],[336,270],[343,294],[347,121],[354,116],[522,1115],[526,980],[550,116],[573,274],[616,610],[622,270],[988,449],[1046,693],[1097,1382],[1205,721],[1242,290],[1354,121],[1373,100],[68,1320],[71,457],[83,607],[98,101],[112,388],[119,111],[268,1289],[271,589],[342,116],[375,766],[436,711],[447,958],[465,53],[552,368],[603,117],[690,1425],[1266,724],[1359,1182],[67,69],[68,368],[82,323],[102,408],[109,491],[111,98],[111,400],[115,925],[264,368],[265,301],[266,116],[275,115],[278,893],[296,1349],[297,674],[302,1273],[303,367],[319,683],[355,1007],[372,1125],[372,1158],[383,97],[482,796],[487,1060],[505,837],[535,556],[571,1202],[615,532],[616,293],[638,273],[807,281],[827,290],[935,595],[1047,76],[1070,115],[1236,270],[32,409],[72,101],[77,1332],[80,85],[83,1038],[84,435],[84,897],[100,331],[102,574],[103,263],[109,528],[299,731],[299,736],[301,101],[301,433],[307,824],[312,1458],[329,288],[332,396],[487,466],[659,911],[705,436],[858,891],[1009,278],[1280,462],[1397,563],[1407,366],[67,493],[78,493],[80,69],[82,79],[101,378],[102,111],[102,266],[102,1330],[110,111],[118,640],[119,101],[119,736],[264,403],[281,695],[287,1429],[315,1185],[321,278],[332,119],[419,893],[519,1002],[532,101],[559,116],[642,284],[729,890],[748,265],[849,541],[888,120],[900,298],[901,276],[1044,115],[1095,115],[1358,323],[1438,406],[32,122],[32,494],[49,53],[77,643],[79,1161],[80,643],[97,869],[98,97],[103,316],[115,1022],[117,282],[120,120],[276,368],[277,264],[300,333],[302,1281],[319,107],[321,273],[337,428],[355,397],[562,110],[574,100],[651,745],[838,281],[855,1449],[931,298],[1560,75],[32,1240],[48,50],[78,1475],[84,67],[85,620],[86,282],[99,556],[102,99],[102,812],[105,295],[115,112],[119,570],[119,870],[265,905],[268,1133],[273,1143],[275,294],[278,592],[302,1363],[307,618],[309,341],[336,115],[380,532],[390,54],[448,330],[519,415],[591,979],[636,115],[875,1345],[1029,323],[1039,1277],[1179,1174],[1410,1515],[1434,275],[1597,270],[32,54],[32,107],[32,1439],[48,53],[55,55],[71,1127],[73,83],[78,707],[79,690],[85,979],[88,82],[97,436],[98,1107],[102,363],[102,750],[104,266],[104,270],[108,112],[109,493],[112,263],[112,628],[119,119],[273,266],[279,274],[286,330],[287,1067],[291,363],[315,330],[332,1546],[333,972],[372,115],[437,1611],[475,341],[504,839],[601,115],[661,1094],[679,1108],[744,328],[748,115],[929,115],[938,278],[1040,341],[1102,48],[1114,1243],[32,356],[67,407],[67,1450],[77,65],[77,101],[97,308],[104,451],[109,590],[111,103],[115,484],[118,294],[119,623],[120,66],[264,1101],[270,537],[273,868],[299,661],[299,703],[300,1123],[309,108],[320,267],[324,102],[325,1007],[331,104],[351,104],[354,509],[399,1272],[414,114],[430,421],[431,110],[435,100],[458,107],[553,715],[603,323],[607,107],[733,691],[812,275],[827,115],[844,1012],[957,282],[981,368],[986,872],[1004,903],[1059,512],[1234,1526],[1312,435],[1447,106],[1472,339],[1608,90],[32,424],[49,52],[51,50],[54,54],[56,48],[65,115],[67,75],[71,1481],[76,65],[79,83],[83,561],[85,78],[86,454],[103,108],[105,266],[105,647],[108,270],[109,673],[109,1527],[109,1529],[112,270],[112,273],[114,673],[115,429],[116,270],[116,330],[116,1364],[120,98],[265,295],[278,1088],[282,281],[294,721],[296,925],[299,825],[307,1131],[310,112],[315,107],[325,1104],[325,1627],[333,289],[334,715],[344,1135],[354,1486],[356,691],[370,290],[376,116],[471,897],[508,308],[531,919],[563,488],[576,453],[604,70],[631,492],[657,579],[748,432],[765,1692],[887,581],[930,308],[1051,569],[1296,1012],[1307,120],[1536,331],[1680,288],[1748,1741],[66,73],[66,674],[80,454],[98,1285],[98,1409],[99,1239],[107,711],[109,121],[110,532],[111,569],[111,1334],[117,334],[262,350],[264,824],[277,278],[282,449],[286,1246],[296,1159],[308,1209],[312,108],[317,885],[326,276],[328,575],[365,121],[382,787],[398,541],[398,1682],[402,298],[434,657],[459,623],[461,278],[465,390],[475,298],[486,110],[486,266],[550,68],[589,121],[667,115],[684,115],[733,1617],[776,1068],[784,784],[1040,634],[1051,264],[1229,610],[1468,274],[1505,99],[32,88],[32,1073],[57,57],[65,100],[66,677],[67,267],[67,1178],[68,525],[77,76],[80,65],[83,1081],[98,834],[98,1381],[99,749],[100,117],[105,289],[110,707],[111,346],[116,108],[117,336],[119,1160],[268,345],[275,1068],[277,597],[287,1155],[287,1238],[290,275],[291,1144],[296,388],[296,538],[297,606],[303,116],[317,115],[329,528],[372,1079],[390,53],[399,1362],[447,555],[465,48],[472,264],[565,822],[626,474],[658,715],[720,1501],[730,780],[737,105],[751,1779],[796,1117],[838,1002],[995,119],[1018,1010],[1037,579],[1090,120],[1341,417],[1512,1183],[57,49],[65,596],[83,67],[83,85],[83,1754],[84,73],[86,1135],[89,1507],[97,562],[98,621],[106,111],[109,773],[114,328],[114,1572],[115,406],[117,263],[117,478],[121,290],[275,109],[279,1187],[296,266],[296,1830],[301,274],[301,492],[302,905],[311,350],[329,885],[335,829],[356,1326],[365,273],[385,105],[408,265],[413,1295],[413,1857],[422,267],[431,78],[442,814],[471,104],[530,115],[531,84],[563,273],[573,850],[599,115],[612,270],[666,602],[689,115],[699,1723],[772,451],[857,1180],[929,282],[957,115],[978,115],[991,298],[996,891],[1056,985],[1109,1344],[1181,868],[1314,112],[1372,487],[1783,1074],[32,313],[32,509],[32,581],[65,1133],[65,1671],[70,750],[73,88],[80,1786],[83,72],[83,665],[84,415],[84,918],[97,647],[97,724],[99,290],[100,98],[103,104],[110,855],[111,102],[114,512],[115,119],[115,688],[118,111],[118,648],[119,865],[275,308],[279,1218],[291,1375],[293,330],[296,1721],[302,301],[302,947],[307,103],[307,1292],[308,673],[321,612],[321,1953],[332,797],[333,100],[334,294],[334,298],[335,266],[345,298],[354,1854],[354,1917],[364,274],[366,330],[375,109],[383,114],[399,631],[407,115],[419,592],[468,101],[553,923],[553,1188],[604,82],[624,1282],[625,111],[693,68],[695,346],[698,797],[713,298],[723,1923],[765,69],[870,265],[895,503],[1027,115],[1192,148],[1346,989],[1479,691],[1525,470],[1565,546],[1823,116],[1922,1892],[49,54],[57,52],[65,68],[65,78],[66,516],[68,665],[73,330],[77,491],[79,68],[80,79],[80,1422],[86,69],[87,78],[89,998],[97,507],[103,274],[103,989],[109,100],[112,540],[115,1328],[116,290],[117,625],[119,317],[120,116],[265,100],[271,1136],[276,284],[277,109],[279,1098],[280,108],[287,435],[287,1245],[299,527],[300,606],[306,1766],[307,102],[313,274],[320,532],[329,265],[329,1106],[336,462],[339,1265],[344,1772],[365,712],[365,1257],[372,580],[384,985],[424,110],[479,773],[517,112],[537,1244],[568,348],[634,115],[763,416],[766,1143],[837,115],[914,100],[936,1261],[948,396],[949,714],[950,99],[986,1443],[1164,1849],[1177,289],[1192,166],[1214,115],[1279,786],[1348,1905],[1394,1316],[1395,115],[1805,2061],[1819,462],[1844,457],[1869,265],[1916,316],[1930,497],[48,51],[51,52],[65,108],[68,349],[73,100],[79,313],[80,84],[82,1568],[83,69],[83,367],[85,693],[86,1976],[97,575],[102,558],[102,1504],[105,281],[105,911],[109,643],[119,719],[120,70],[264,809],[283,261],[291,1377],[293,116],[296,629],[296,1141],[297,516],[300,327],[313,298],[320,493],[331,285],[333,103],[334,1060],[336,345],[409,84],[411,1156],[414,290],[427,1824],[458,323],[482,98],[553,1677],[621,104],[698,280],[701,1244],[895,1255],[915,282],[949,115],[962,1965],[1016,923],[1110,446],[1204,461],[1367,115],[1648,445],[2029,944],[2072,621],[2102,1454],[32,90],[49,390],[50,390],[69,756],[70,363],[71,469],[77,73],[79,1083],[82,111],[83,1020],[84,104],[84,1496],[85,77],[85,1718],[88,1543],[99,1024],[105,478],[109,265],[109,1331],[112,1123],[114,341],[117,568],[118,270],[264,1131],[264,1223],[267,724],[270,575],[271,284],[273,580],[276,712],[276,1257],[276,1732],[278,494],[293,1156],[300,351],[300,628],[307,110],[329,347],[329,1580],[329,1598],[342,2046],[391,285],[399,479],[409,69],[411,1463],[427,1793],[434,265],[461,334],[553,294],[619,1325],[659,1517],[665,549],[690,65],[738,262],[744,666],[776,1103],[788,115],[812,109],[860,99],[874,1061],[918,69],[1045,1848],[1184,115],[1196,298],[1208,370],[1251,341],[1256,451],[1304,1476],[1665,542],[1714,898],[1776,278],[1808,87],[1963,274],[1982,2196],[1983,368],[2138,488],[2152,2110],[32,55],[65,66],[66,85],[68,1585],[69,88],[70,265],[70,1221],[70,1704],[71,1206],[73,1246],[76,79],[76,101],[76,737],[80,406],[84,111],[87,1616],[89,461],[99,105],[99,288],[99,1178],[99,1218],[103,392],[103,1119],[104,340],[109,1175],[110,267],[111,293],[116,275],[120,795],[121,657],[264,99],[264,542],[268,869],[279,1313],[288,503],[294,1247],[296,1081],[299,332],[300,705],[300,1503],[307,1287],[309,290],[317,691],[321,1075],[325,1011],[327,115],[329,491],[333,470],[342,1351],[380,335],[384,1386],[398,837],[399,112],[411,1870],[412,474],[450,339],[454,2068],[468,769],[470,278],[471,72],[517,1421],[537,100],[581,115],[617,115],[622,2150],[659,403],[744,612],[768,2258],[842,1300],[892,1790],[952,1467],[981,513],[1024,266],[1053,275],[1055,433],[1065,595],[1162,101],[1193,657],[1259,903],[1291,276],[1305,583],[1335,115],[1371,746],[1406,264],[1730,595],[1765,110],[1863,2200],[1889,1003],[1896,2040],[2101,1099],[2223,1020],[32,806],[32,826],[48,55],[56,56],[66,89],[68,469],[71,747],[71,1955],[73,303],[76,998],[77,433],[77,1175],[79,453],[80,2064],[82,97],[83,1141],[97,446],[99,719],[100,270],[102,850],[108,100],[108,288],[108,634],[110,364],[111,263],[116,1238],[118,103],[121,109],[263,290],[270,1465],[282,1652],[287,332],[287,897],[295,699],[302,521],[307,1353],[314,112],[316,115],[317,652],[325,397],[327,276],[339,319],[344,734],[345,446],[365,558],[372,1868],[380,725],[413,905],[421,100],[438,101],[470,121],[472,108],[515,934],[555,303],[571,115],[622,1537],[630,703],[646,115],[758,1024],[760,48],[797,97],[804,115],[817,1275],[830,833],[831,274],[840,1775],[901,1031],[921,575],[930,639],[939,462],[949,319],[986,2006],[990,417],[1005,2005],[1053,348],[1085,593],[1093,115],[1096,575],[1152,103],[1271,2189],[1314,385],[1369,691],[1545,478],[1575,288],[1615,301],[1980,76],[2008,392],[2055,513],[2159,478],[2364,2304],[32,56],[32,1675],[49,56],[49,57],[50,51],[57,48],[65,402],[66,1115],[72,451],[76,740],[77,409],[78,83],[83,420],[83,1269],[84,1866],[86,718],[97,310],[99,1521],[99,1836],[101,647],[102,851],[109,623],[111,276],[112,612],[117,376],[117,923],[120,69],[264,114],[265,340],[265,673],[268,333],[270,118],[275,1103],[275,2396],[276,121],[276,661],[279,1502],[280,1158],[283,32],[283,350],[284,308],[286,1104],[289,467],[297,97],[297,1755],[300,612],[307,116],[308,340],[328,417],[329,2124],[339,121],[342,69],[342,433],[342,678],[342,1038],[344,111],[345,796],[355,2199],[372,112],[377,68],[384,2404],[387,1445],[413,521],[413,1510],[422,499],[440,104],[448,854],[455,340],[468,69],[487,1099],[492,370],[492,2395],[508,639],[535,1386],[553,1729],[568,532],[589,521],[591,80],[633,683],[633,743],[637,303],[659,2323],[716,115],[723,1984],[737,714],[826,2121],[841,115],[858,862],[871,53],[900,115],[901,2231],[922,370],[947,1172],[982,1876],[1011,500],[1030,958],[1041,1074],[1056,556],[1164,915],[1168,115],[1276,1247],[1278,700],[1288,115],[1430,766],[1432,712],[1444,436],[1498,773],[1528,485],[1583,280],[1719,115],[1845,1737],[1859,916],[1949,944],[1978,111],[2028,592],[2053,343],[2054,872],[2160,1882],[2247,1213],[2302,110],[2303,773],[2346,1283],[2349,115],[2378,2454],[32,696],[67,72],[73,65],[80,111],[82,715],[84,82],[87,570],[97,276],[98,116],[100,527],[102,112],[102,1377],[102,1562],[103,290],[104,1419],[105,694],[108,1140],[111,120],[112,328],[113,608],[115,1159],[116,99],[116,282],[116,1180],[119,105],[120,774],[121,112],[268,103],[268,945],[275,116],[275,555],[279,1124],[280,2067],[282,107],[287,886],[291,321],[291,1095],[291,1562],[291,1708],[292,99],[293,1463],[294,602],[295,936],[296,503],[299,1098],[300,711],[300,1938],[313,102],[319,284],[320,415],[325,1906],[336,121],[336,294],[337,350],[348,1177],[355,697],[372,116],[380,1419],[383,911],[399,99],[413,330],[426,364],[472,1956],[484,1163],[489,2097],[515,72],[531,69],[535,985],[568,1867],[571,710],[587,284],[589,266],[624,115],[636,100],[650,115],[671,115],[702,341],[705,1009],[723,600],[729,370],[758,2399],[771,2113],[799,1290],[829,298],[871,54],[890,301],[927,546],[938,376],[940,290],[948,289],[962,376],[983,115],[997,1925],[1030,555],[1041,1736],[1054,115],[1196,323],[1198,115],[1252,634],[1328,2044],[1353,549],[1405,341],[1416,1487],[1471,714],[1539,348],[1557,555],[1807,575],[1862,2599],[1914,1690],[1998,1239],[2002,2103],[2038,1586],[2049,513],[2078,2416],[2133,348],[2351,2012],[2435,1635],[2466,513],[2542,883],[2588,600],[2614,1084],[32,298],[32,1629],[51,53],[55,53],[56,52],[65,98],[66,606],[70,1144],[70,1268],[71,416],[71,2014],[78,466],[81,117],[83,2280],[84,1155],[85,82],[85,278],[89,741],[97,1010],[99,281],[99,1187],[100,102],[105,819],[107,105],[112,119],[112,1886],[114,99],[116,2016],[117,120],[117,683],[263,270],[264,109],[266,111],[274,1890],[276,1259],[279,1521],[280,1762],[286,109],[288,1932],[291,855],[292,309],[295,652],[296,867],[296,1022],[300,105],[314,1768],[314,2529],[325,849],[328,326],[334,923],[344,331],[352,780],[355,849],[366,2525],[383,1778],[390,57],[392,266],[402,290],[409,70],[411,534],[411,713],[419,109],[419,494],[431,67],[440,1898],[443,600],[449,266],[471,843],[480,381],[527,316],[553,966],[553,2365],[581,290],[645,420],[645,780],[651,65],[651,101],[677,69],[729,284],[799,1465],[811,100],[811,547],[819,478],[844,2034],[866,115],[869,266],[909,462],[931,2009],[935,563],[1016,1188],[1035,323],[1078,115],[1080,110],[1085,421],[1092,319],[1121,316],[1190,2073],[1215,2033],[1234,618],[1274,341],[1348,282],[1378,282],[1398,512],[1406,705],[1495,121],[1621,119],[1874,274],[1894,2637],[1942,634],[1972,404],[2015,264],[2026,406],[2282,115],[2442,2249],[2486,1459],[2561,112],[2617,763],[2634,1261],[2693,2724],[48,56],[50,49],[57,53],[66,1381],[67,524],[67,1313],[67,2132],[68,2085],[73,68],[73,77],[76,527],[78,84],[78,1799],[79,102],[80,745],[82,315],[83,79],[83,117],[83,925],[87,1678],[99,516],[99,1581],[100,281],[102,1535],[103,114],[105,714],[108,340],[110,450],[110,493],[114,315],[114,1659],[115,2319],[116,112],[116,115],[116,1761],[117,403],[118,323],[119,110],[263,331],[263,589],[264,1059],[265,115],[265,276],[265,323],[265,1510],[270,891],[270,1290],[271,1023],[273,115],[273,116],[278,101],[279,108],[279,2423],[283,441],[287,114],[287,1023],[287,1496],[287,2100],[288,1255],[291,1221],[291,1268],[291,1555],[292,1713],[292,2784],[293,115],[296,112],[296,915],[297,2412],[307,100],[309,547],[312,2684],[323,1743],[329,643],[329,1175],[337,32],[344,1452],[345,295],[356,1287],[372,537],[380,719],[380,1097],[383,1517],[383,2224],[390,56],[419,403],[419,2105],[430,273],[440,363],[465,54],[472,1957],[479,319],[504,1021],[520,2714],[526,998],[555,276],[569,1655],[577,1589],[577,1745],[580,103],[591,620],[615,1931],[622,294],[626,2522],[637,1011],[644,2782],[729,1961],[733,885],[776,308],[808,1483],[810,614],[887,290],[917,2070],[927,115],[995,623],[995,2141],[1004,275],[1016,294],[1045,77],[1075,99],[1086,1589],[1086,2449],[1091,381],[1110,120],[1169,975],[1230,1570],[1251,485],[1341,266],[1400,492],[1426,2855],[1444,561],[1453,1269],[1461,298],[1474,588],[1482,2288],[1508,76],[1553,883],[1625,115],[1626,115],[1650,903],[1891,1912],[1895,868],[2019,121],[2108,1715],[2267,673],[2307,623],[2379,109],[2513,101],[49,55],[50,54],[53,1994],[57,55],[65,289],[65,1228],[65,1571],[67,444],[68,70],[68,331],[69,2736],[70,97],[70,498],[77,2001],[78,69],[78,1641],[79,70],[79,76],[86,2185],[97,120],[98,301],[102,315],[102,336],[102,1221],[102,1268],[103,1127],[110,415],[112,711],[112,1503],[114,117],[115,501],[115,2007],[115,2320],[116,336],[116,1067],[116,1155],[116,2172],[117,620],[118,718],[120,67],[120,121],[121,998],[264,276],[265,1295],[265,1738],[267,115],[268,414],[270,117],[279,308],[283,262],[284,348],[287,2309],[287,2533],[290,115],[291,1504],[291,1628],[292,1851],[296,109],[296,110],[296,301],[296,309],[296,429],[297,327],[297,1285],[297,1991],[298,546],[300,121],[307,1428],[310,294],[312,351],[312,1877],[312,2568],[319,275],[332,289],[333,101],[334,1188],[342,2042],[352,1335],[352,1911],[356,809],[357,1399],[365,567],[365,1260],[365,1559],[372,1762],[384,1835],[384,2934],[399,2289],[413,1417],[413,2283],[418,359],[419,938],[419,1088],[424,115],[430,1136],[431,945],[431,1757],[449,2311],[452,2128],[454,1084],[460,350],[468,2509],[468,2818],[470,116],[475,290],[480,859],[483,402],[491,330],[503,103],[508,396],[512,101],[528,266],[548,115],[564,581],[571,2418],[588,2642],[603,290],[604,83],[612,115],[622,1140],[641,2571],[656,786],[668,417],[705,478],[733,264],[735,265],[756,298],[762,115],[765,1989],[772,290],[796,492],[797,499],[799,2263],[804,294],[818,298],[819,1213],[826,1988],[838,415],[858,2530],[880,592],[884,290],[888,719],[888,2743],[890,107],[935,1511],[952,539],[1090,263],[1173,115],[1200,563],[1274,298],[1293,290],[1305,376],[1357,683],[1380,1717],[1471,115],[1498,370],[1528,341],[1549,1549],[1552,3024],[1553,1064],[1554,266],[1576,101],[1615,1119],[1623,2584],[1637,290],[1645,907],[1649,1452],[1657,2051],[1691,1707],[1750,1577],[1810,1300],[1871,2772],[1887,695],[1887,2145],[1928,2816],[1970,298],[1999,341],[2131,3015],[2135,589],[2153,1959],[2215,2879],[2240,281],[2245,115],[2326,298],[2451,2912],[2497,115],[2638,115],[2691,2079],[2692,2881],[2722,1408],[2898,492],[2900,732],[2993,1606],[2995,3086],[32,75],[32,677],[48,52],[55,54],[57,50],[65,652],[67,769],[67,1701],[69,2506],[70,67],[70,73],[70,459],[71,653],[73,2004],[79,385],[83,867],[86,401],[87,825],[97,433],[99,1107],[100,1160],[101,404],[102,100],[102,313],[103,116],[103,264],[104,980],[104,1668],[105,683],[110,115],[110,1641],[112,333],[112,385],[114,909],[115,388],[115,629],[117,305],[118,449],[118,661],[118,1033],[121,555],[121,1423],[264,103],[264,1326],[268,806],[270,99],[275,403],[278,109],[278,3124],[280,1125],[280,1619],[282,115],[286,2430],[287,266],[287,503],[288,290],[289,120],[290,1243],[291,1535],[292,1161],[293,580],[296,119],[296,1763],[296,2331],[297,1409],[297,2408],[300,388],[300,417],[306,1255],[307,2062],[312,2343],[333,280],[335,3109],[342,1081],[352,717],[352,1272],[355,1903],[356,1101],[356,1292],[371,2461],[372,102],[372,1350],[372,1619],[376,353],[380,2991],[394,2291],[398,2308],[413,2766],[427,2892],[440,111],[440,745],[448,308],[456,1757],[461,1935],[468,65],[468,822],[471,79],[471,1067],[479,298],[483,1571],[503,486],[508,121],[510,3121],[519,2274],[523,122],[523,988],[537,99],[550,303],[553,3001],[565,2168],[571,1177],[591,278],[595,282],[597,99],[617,1558],[622,451],[626,1185],[630,101],[633,293],[663,115],[693,2753],[701,99],[730,1362],[733,2431],[739,104],[776,2514],[826,2610],[827,266],[828,32],[833,115],[841,298],[861,1940],[872,657],[880,1088],[884,936],[912,1655],[914,1163],[930,597],[931,276],[969,766],[977,115],[1018,275],[1046,84],[1050,115],[1090,564],[1190,958],[1193,2607],[1204,323],[1216,1742],[1223,316],[1265,282],[1331,266],[1391,953],[1461,323],[1469,115],[1480,1041],[1483,927],[1493,115],[1524,115],[1552,1710],[1556,2090],[1576,290],[1630,1743],[1709,1487],[1716,602],[1791,319],[1801,1758],[1832,298],[1840,290],[1860,1108],[1879,470],[1941,449],[1951,1408],[1968,370],[2074,3038],[2092,319],[2123,2299],[2139,2904],[2208,3067],[2212,1699],[2236,1570],[2413,97],[2432,115],[2515,528],[2534,528],[2550,2559],[2600,1724],[2702,115],[2745,3266],[2757,266],[2796,489],[2798,115],[2873,2759],[2960,541],[3106,267],[3137,298],[3173,3262],[3180,3284],[3183,1340],[3210,274],[32,74],[32,395],[53,48],[65,2177],[66,834],[67,70],[67,1502],[70,1375],[71,2681],[72,532],[76,76],[76,1140],[77,374],[77,3239],[79,66],[80,1649],[82,532],[83,1813],[83,2140],[85,1873],[86,82],[87,101],[88,1812],[99,321],[99,1313],[99,2010],[100,284],[100,301],[101,1877],[102,509],[103,323],[104,351],[104,532],[105,264],[107,270],[107,2287],[108,116],[108,301],[110,101],[110,469],[110,1351],[112,1106],[112,1136],[112,2649],[114,108],[114,715],[114,1822],[115,103],[115,2114],[116,294],[119,265],[119,332],[268,888],[272,369],[274,475],[279,1581],[280,647],[281,944],[282,321],[282,394],[282,547],[284,293],[287,101],[287,896],[288,2774],[291,101],[291,343],[291,1423],[296,1269],[296,1491],[296,2140],[299,870],[300,104],[302,1738],[306,1939],[307,1511],[307,3193],[312,2587],[313,435],[313,579],[323,1018],[325,1903],[328,525],[329,642],[329,1787],[342,84],[342,469],[344,1000],[352,2176],[356,276],[356,2286],[356,3107],[363,282],[365,574],[365,606],[371,1873],[394,3007],[409,497],[411,523],[412,1890],[422,1522],[427,2353],[431,596],[440,264],[440,267],[443,1492],[456,69],[462,967],[472,111],[472,1127],[480,110],[480,2910],[487,819],[515,65],[522,3267],[524,290],[570,1066],[571,1445],[573,69],[587,266],[600,2983],[604,3367],[622,1622],[626,2927],[630,1616],[690,2075],[698,3023],[704,1283],[705,561],[723,70],[730,420],[744,1075],[751,115],[766,334],[784,261],[799,1791],[804,1554],[841,341],[858,100],[887,298],[890,341],[898,989],[935,2315],[962,1182],[995,569],[1039,1674],[1071,115],[1167,274],[1173,936],[1229,2595],[1271,435],[1304,1742],[1338,84],[1380,101],[1387,115],[1392,1083],[1448,115],[1480,3379],[1482,3378],[1485,1674],[1525,647],[1583,746],[1595,298],[1610,842],[1624,634],[1643,1542],[1651,115],[1804,1812],[1843,1622],[1856,323],[1888,714],[1891,1500],[1920,266],[1928,3395],[1945,290],[1960,2410],[2087,115],[2123,3301],[2157,1401],[2226,115],[2248,115],[2297,3208],[2316,298],[2347,2694],[2372,2445],[2424,1559],[2426,2974],[2429,2968],[2434,2761],[2462,298],[2481,316],[2526,115],[2541,323],[2645,531],[2717,1811],[2924,3012],[3004,724],[3125,290],[3141,2142],[3195,388],[3198,512],[3203,2143],[3221,2740],[3233,1074],[3237,2000],[3285,2488],[3340,353],[3342,3380],[3362,266],[3368,310],[3476,2345],[3483,331],[32,389],[32,1227],[32,1663],[32,2107],[50,52],[55,56],[67,1098],[67,1187],[67,2496],[67,3404],[69,988],[70,1116],[70,1535],[70,1628],[72,69],[72,1989],[73,1228],[73,1325],[77,267],[77,469],[77,549],[77,590],[77,2666],[78,2899],[79,114],[79,470],[79,677],[80,288],[80,1902],[80,2232],[80,2572],[80,3448],[85,112],[85,677],[85,690],[87,79],[87,392],[87,2540],[97,78],[97,101],[97,102],[97,275],[98,685],[98,951],[99,341],[99,2099],[99,2501],[100,100],[102,1144],[103,98],[103,103],[103,282],[103,417],[104,273],[104,335],[106,700],[109,351],[110,116],[110,2829],[111,385],[111,1969],[112,2146],[115,310],[115,420],[115,538],[115,991],[115,1763],[116,285],[116,450],[118,417],[119,294],[119,527],[120,68],[120,100],[120,997],[240,157],[263,273],[264,654],[264,1292],[265,1342],[265,1417],[268,109],[268,403],[271,2947],[273,1066],[275,3126],[276,474],[278,403],[279,97],[279,117],[279,3037],[280,1079],[280,1290],[281,370],[282,805],[284,116],[285,115],[289,1245],[292,115],[292,955],[292,2313],[293,383],[296,420],[299,803],[301,435],[302,1417],[302,3056],[307,391],[307,1101],[307,1488],[307,1730],[307,2284],[312,2147],[313,336],[314,1969],[315,319],[315,575],[315,1012],[317,120],[320,3016],[323,99],[325,264],[327,112],[329,1331],[329,1768],[330,761],[331,111],[336,105],[336,290],[339,102],[342,72],[342,266],[351,319],[351,629],[352,665],[356,1059],[360,264],[365,612],[368,370],[372,99],[377,1700],[380,816],[380,3495],[383,819],[384,1818],[419,654],[427,1140],[431,115],[433,3584],[436,627],[442,107],[448,3171],[449,120],[454,82],[471,467],[471,777],[471,2203],[471,3443],[472,114],[480,536],[480,1073],[486,1710],[487,280],[488,100],[497,70],[503,109],[507,1001],[508,264],[535,1835],[544,290],[544,3633],[545,48],[551,1394],[552,516],[567,2402],[573,1708],[577,110],[588,927],[600,3066],[604,110],[615,1867],[634,121],[637,1906],[650,653],[651,121],[659,276],[659,289],[661,319],[665,2646],[676,3597],[699,323],[765,3664],[784,428],[815,115],[827,298],[830,2284],[873,2662],[877,530],[880,654],[884,115],[887,3428],[902,1390],[915,298],[948,2260],[955,353],[956,3542],[1035,3097],[1053,3616],[1065,108],[1065,563],[1085,2795],[1086,692],[1087,115],[1091,2452],[1110,280],[1162,3295],[1183,115],[1186,115],[1205,103],[1250,115],[1250,1516],[1259,1374],[1265,683],[1274,115],[1296,829],[1340,89],[1372,562],[1378,100],[1380,290],[1391,967],[1427,290],[1446,115],[1485,1277],[1506,115],[1509,115],[1513,696],[1538,2660],[1614,1008],[1689,115],[1705,83],[1780,275],[1787,3306],[1821,1640],[1825,351],[1833,341],[1861,3413],[1881,115],[1888,319],[1919,325],[1927,65],[1939,1668],[1952,115],[1997,3729],[2031,549],[2039,115],[2136,687],[2161,1147],[2206,84],[2219,105],[2237,115],[2238,319],[2298,97],[2333,2117],[2428,484],[2463,549],[2531,2651],[2546,2062],[2555,1454],[2565,773],[2622,1733],[2688,417],[2706,3119],[2707,316],[2926,1129],[3057,746],[3102,2045],[3108,3175],[3127,661],[3145,115],[3160,295],[3178,531],[3226,298],[3283,1030],[3311,478],[3351,3735],[3371,1702],[3387,282],[3527,1342],[3529,2099],[3537,266],[3586,114],[3625,628],[3649,3128],[3660,497],[3710,1343],[3715,546],[3771,368],[32,57],[32,446],[32,1558],[32,2573],[51,48],[53,54],[56,51],[57,54],[65,67],[65,75],[65,677],[70,2590],[71,69],[71,73],[71,1392],[72,1174],[72,2041],[73,79],[76,270],[77,3792],[78,111],[79,79],[79,1241],[80,333],[80,416],[80,934],[80,946],[82,368],[82,1659],[83,73],[83,814],[83,2357],[84,1023],[84,1245],[84,2122],[84,2172],[85,859],[97,336],[97,403],[97,945],[98,99],[98,108],[98,313],[98,1991],[99,117],[99,3201],[100,265],[100,652],[102,855],[103,653],[105,335],[105,397],[106,112],[107,301],[108,417],[108,2146],[109,275],[110,1799],[111,426],[111,666],[111,1574],[112,288],[112,407],[112,2962],[114,116],[114,368],[116,1964],[117,303],[118,290],[119,100],[122,121],[122,711],[263,417],[264,568],[264,2166],[265,1000],[265,3643],[266,114],[268,3860],[270,819],[273,1446],[274,330],[276,612],[276,3855],[278,654],[278,938],[280,118],[280,891],[280,2373],[280,2615],[280,3863],[281,266],[283,1032],[287,2122],[288,121],[288,839],[288,1136],[289,282],[289,1023],[291,112],[291,309],[291,812],[291,2467],[291,2590],[292,3869],[294,2558],[296,281],[296,407],[296,462],[296,937],[296,2007],[296,2606],[297,1107],[297,2676],[299,449],[300,108],[300,1886],[300,1960],[300,3054],[301,334],[302,118],[302,2769],[307,276],[308,1594],[309,3330],[313,285],[314,3895],[318,920],[320,607],[321,2081],[326,647],[327,121],[327,386],[328,290],[329,2857],[330,664],[331,407],[339,1058],[342,79],[342,111],[342,946],[342,1129],[345,323],[348,97],[348,101],[351,462],[351,1094],[351,3758],[352,301],[356,2166],[360,278],[372,275],[372,1260],[372,2373],[372,3027],[377,1610],[379,115],[385,115],[393,528],[394,1103],[399,1719],[407,100],[411,3143],[413,1822],[413,2377],[422,2243],[422,3282],[431,652],[431,2177],[433,3834],[440,493],[440,1502],[449,281],[450,115],[451,3681],[471,3535],[472,492],[479,1094],[480,88],[483,426],[488,108],[494,1151],[505,872],[505,3506],[508,1875],[510,332],[511,3853],[515,3101],[517,109],[523,1011],[524,266],[528,298],[532,266],[533,290],[535,3029],[542,1631],[551,79],[566,278],[571,947],[571,3217],[580,396],[581,294],[598,513],[600,115],[604,2842],[609,116],[611,1790],[626,1944],[627,370],[633,385],[641,2274],[646,774],[651,3516],[657,1399],[658,3549],[666,270],[677,1995],[699,341],[720,1009],[723,1607],[730,1009],[744,2081],[745,777],[753,1043],[756,3872],[777,2004],[800,3677],[810,330],[811,683],[821,298],[838,3005],[857,3321],[860,450],[860,2056],[875,421],[877,539],[877,2011],[877,3546],[888,564],[907,115],[921,746],[934,83],[974,2321],[975,1715],[981,883],[981,1064],[991,3825],[997,454],[1004,3851],[1017,2532],[1041,2297],[1045,2532],[1047,1316],[1049,115],[1071,1577],[1082,4022],[1085,270],[1091,3667],[1100,370],[1120,588],[1179,524],[1193,2056],[1207,115],[1215,276],[1216,1070],[1228,1206],[1241,1340],[1241,3526],[1250,1008],[1256,267],[1278,3670],[1279,1866],[1304,1070],[1312,1099],[1334,115],[1346,513],[1356,3347],[1360,546],[1400,1548],[1464,115],[1478,68],[1490,115],[1490,266],[1516,266],[1522,791],[1559,3166],[1582,298],[1601,119],[1604,1604],[1609,3490],[1645,2340],[1650,805],[1667,298],[1693,115],[1696,1696],[1698,4043],[1770,115],[1780,2877],[1807,100]],"version":2}

In [ ]:
%%writefile kani_advanced.py
"""Kani Advanced: Kani's own transformer, pretrained from scratch on Kaggle's free GPUs.

The architecture and tokenizer are exactly the local Kani's (kelus/model.py, data/big-v1/tokenizer.json):
pre-LayerNorm blocks, rotary positions, a 4x GELU feed-forward and tied embeddings, so the exported
kani-advanced.npz loads straight into `kelus`. No pretrained weights: it learns from public text,
FineWeb-Edu and Cosmopedia v2, a new slice of it every Kaggle session.

    python kani_advanced.py prepare --settings settings.json    # download and tokenize this session's text
    torchrun --nnodes=1 --nproc_per_node=2 --master_addr=127.0.0.1 --master_port=29500 kani_advanced.py train --settings settings.json
    python kani_advanced.py sample --settings settings.json --prompt "The moon is"
    python kani_advanced.py export --settings settings.json     # kani-advanced.npz from checkpoint.pt
    python kani_advanced.py retokenize --settings settings.json --checkpoint old/checkpoint.pt --old-tokenizer old.json
    python kani_advanced.py grams --tests DIR --out contamination.npy   # the test sets' 13-word runs, never to train on
    python kani_advanced.py grow --settings settings.json --base kani-advanced.npz --progress progress.json   # a deeper run

Multilingual runs (BPE pattern version 3) give each source a `share` of the session's tokens, a `filter` that keeps
documents in their language's script, and a held-out `validation_file`; `retokenize` moves an English checkpoint
to the multilingual vocabulary, keeping everything it learned. Shares also mix English with code. A source's
`revision` reads another branch of its dataset, such as Hugging Face's parquet copy of a JSON dataset
('refs/convert/parquet'), and its `column` names the field holding the text ('text' unless set; 'code' for code).
"""
import functools
import argparse
import hashlib
from contextlib import nullcontext
import glob
import itertools
import json
import math
import os
from pathlib import Path
import re
import shutil
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

PAD, BOS, EOS, USER, ASSISTANT = range(256, 261)

LATIN, HANGUL = "A-Za-z\u00c0-\u00d6\u00d8-\u00f6\u00f8-\u024f\u1e00-\u1eff", "\uac00-\ud7a3\u1100-\u11ff\u3130-\u318f"
KANA, HAN = "\u3040-\u309f\u30a0-\u30ff\u31f0-\u31ff", "\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff"
PATTERNS = {2: re.compile(r"```[A-Za-z]*| ?[A-Za-z]+| ?[0-9]+|\s+|[^A-Za-z0-9\s]"),
            3: re.compile(f"```[A-Za-z]*| ?[{LATIN}]+| ?[{HANGUL}]+|[{KANA}]+|[{HAN}]+| ?[0-9]+|\\s+|[^\\s]")}

DEFAULTS = {
    'run_dir': '/kaggle/working/kani-advanced',
    'input_root': '/kaggle/input',
    'data_dir': '/tmp/kani-advanced-data',
    'tokenizer': 'kani_tokenizer.json',
    'width': 1024, 'layers': 16, 'heads': 16, 'context': 1024,
    'target_tokens': 4_000_000_000,
    'session_tokens': 1_200_000_000,
    'batch_tokens': 262_144,
    'micro_batch': 4,
    'lr': 4e-4, 'min_lr_ratio': 0.1, 'warmup_tokens': 100_000_000, 'decay_fraction': 0.2,
    'weight_decay': 0.1,
    'hours': 10.0,
    'checkpoint_minutes': 30,
    'eval_every': 250, 'eval_batches': 16,
    'validation_documents': 4000,
    'sources': [
        {'repo': 'HuggingFaceFW/fineweb-edu', 'prefix': 'sample/10BT/', 'per_session': 1},
        {'repo': 'HuggingFaceTB/smollm-corpus', 'prefix': 'cosmopedia-v2/', 'per_session': 1},
    ],
    'gradient_checkpointing': False,
    'compile': False,
    'seed': 2041,
    'continue_from_previous': False,
    'workers': 0,
    'rewarm_tokens': 50_000_000,
    'decontaminate': False,
}


def load_settings(path):
    settings = {**DEFAULTS, **json.loads(Path(path).read_text())}
    w, h = settings['width'], settings['heads']

    if not (16 <= settings['context'] <= 2048 and 8 <= w <= 1024 and 1 <= settings['layers'] <= 24
            and w % h == 0 and (w // h) % 2 == 0):
        raise SystemExit('Kani needs context <= 2048, width <= 1024, layers <= 24, and an even head width')
    return settings




class Tokenizer:
    """kelus.tokenizer.BPETokenizer (versions 2 and 3), with an unbounded per-piece cache for speed."""

    def __init__(self, merges, version=2):
        self.version, self.pattern = version, PATTERNS[version]
        self.merges = [tuple(pair) for pair in merges]
        self.pieces = [bytes([i]) for i in range(256)] + [b''] * 5
        self.ranks = {}
        for rank, (a, b) in enumerate(self.merges):
            self.ranks[(a, b)] = (rank, len(self.pieces))
            self.pieces.append(self.pieces[a] + self.pieces[b])
        self.vocab_size = len(self.pieces)
        self.cache = {}

    @classmethod
    def load(cls, path):
        value = json.loads(Path(path).read_text())
        if value.get('kind') != 'bpe' or value.get('version') not in PATTERNS:
            raise SystemExit(f'{path} is not a Kani BPE tokenizer (version 2 or 3)')
        return cls(value['merges'], value['version'])

    def encode_piece(self, piece):
        cached = self.cache.get(piece)
        if cached is not None:
            return cached
        sequence = list(piece.encode('utf-8'))
        while len(sequence) > 1:
            best = None
            for pair in zip(sequence, sequence[1:]):
                rank = self.ranks.get(pair)
                if rank is not None and (best is None or rank < best[0]):
                    best = (rank, pair)
            if best is None:
                break
            (_, token), (a, b) = best
            merged, i = [], 0
            while i < len(sequence):
                if i + 1 < len(sequence) and sequence[i] == a and sequence[i + 1] == b:
                    merged.append(token)
                    i += 2
                else:
                    merged.append(sequence[i])
                    i += 1
            sequence = merged
        if len(self.cache) > 4_000_000:
            self.cache.clear()
        self.cache[piece] = sequence
        return sequence

    def encode(self, text):
        return [token for piece in self.pattern.findall(text) for token in self.encode_piece(piece)]

    def decode(self, tokens):
        return b''.join(self.pieces[t] for t in tokens if 0 <= t < self.vocab_size).decode('utf-8', errors='replace')

    def to_dict(self):
        return {'kind': 'bpe', 'merges': [list(pair) for pair in self.merges], 'version': self.version}


_worker = None


def fast_tokenizer(path):
    """The Rust tokenizer (kaggle/kani_tokenize: the same tokens, ~4.5x faster), or None when it is not installed."""
    try:
        from kani_tokenize import Tokenizer as Rust  # kaggle/kani_tokenize's source folder alone imports, but has no Tokenizer
    except ImportError:
        return None
    value = json.loads(Path(path).read_text())
    return Rust([tuple(pair) for pair in value['merges']], value['version'])


def _start_worker(path, banned=None):
    global _worker, _banned
    _worker = fast_tokenizer(path) or Tokenizer.load(path)
    _banned = np.load(banned) if banned else np.zeros(0, dtype=np.uint64)


# The test sets Kani is measured on (kelus/standard_eval.py: HellaSwag, ARC, PIQA, Winogrande, LAMBADA) are never
# trained on: a document sharing a run of 13 words with one of them is dropped before tokenizing, as GPT-3 did.
# contamination.npy (sorted hashes of those runs) comes from the `grams` command; prepare looks for it in run_dir, then
# beside an attached checkpoint, and stops when `decontaminate` is set and it is missing.
GRAM, _WORDS = 13, re.compile(r'[a-z0-9]+')
_banned = np.zeros(0, dtype=np.uint64)


@functools.lru_cache(maxsize=1 << 20)
def _word_hash(word):
    return int.from_bytes(hashlib.blake2b(word.encode(), digest_size=8).digest(), 'little')


def gram_hashes(text):
    """A 64-bit hash of every run of 13 lower-cased words (letters and digits only) in `text`."""
    words = _WORDS.findall(text.lower())
    count = len(words) - GRAM + 1
    if count < 1:
        return np.zeros(0, dtype=np.uint64)
    ids = np.fromiter(map(_word_hash, words), dtype=np.uint64, count=len(words))
    out = np.zeros(count, dtype=np.uint64)
    for k in range(GRAM):  # wraps at 2**64, the same on every machine
        out = out * np.uint64(1099511628211) + ids[k:k + count]
    return out


def contaminated(text, banned):
    grams = gram_hashes(text)
    if not len(grams) or not len(banned):
        return False
    at = np.minimum(np.searchsorted(banned, grams), len(banned) - 1)
    return bool((banned[at] == grams).any())


def benchmark_texts(folder):
    """Every question with each of its answers, from kelus.standard_eval's test files in `folder`."""
    folder = Path(folder)
    lines = lambda name: [json.loads(line) for line in (folder / name).read_text(encoding='utf-8').splitlines() if line.strip()]
    for row in lines('hellaswag_val.jsonl'):
        yield from (f"{row['activity_label']}: {row['ctx']} {ending}" for ending in row['endings'])
    for name in ('arc_easy_test.jsonl', 'arc_challenge_test.jsonl'):
        for row in lines(name):
            yield from (f"{row['question']} {choice}" for choice in row['choices']['text'])
    for row in lines('piqa_valid.jsonl'):
        yield from (f"{row['goal']} {row['sol1']}", f"{row['goal']} {row['sol2']}")
    for row in lines('winogrande_val.jsonl'):
        yield from (row['sentence'].replace('_', row['option1']), row['sentence'].replace('_', row['option2']))
    for row in lines('lambada_test.jsonl'):
        yield row['text']


def write_grams(folder, out):
    grams = np.unique(np.concatenate([gram_hashes(text) for text in benchmark_texts(folder)]))
    np.save(out, grams)
    print(f'{len(grams):,} distinct 13-word runs from the test sets in {folder} -> {out}')



PAIRS = ('们們 这這 说說 时時 国國 会會 来來 对對 发發 后後 个個 为為 学學 经經 动動 现現 机機 开開 关關 长長 问問 见見 进進 实實 两兩 车車 东東 门門 '
         '应應 电電 气氣 头頭 从從 书書 难難 爱愛 让讓 认認 识識 读讀 写寫 听聽 买買 卖賣 钱錢 银銀 医醫 药藥 话話 语語 请請 谢謝 体體 题題 华華 业業 '
         '产產 网網 线線 结結 给給 红紅 绿綠 鸟鳥 鱼魚 马馬 龙龍 风風 飞飛 云雲 万萬 与與 区區 级級 历歷 战戰 农農 观觀 欢歡 乐樂 节節 质質 资資 '
         '务務 处處 员員 种種 总總 统統 系係 组組 织織 设設 计計 论論 议議 记記 讲講 该該 报報 场場 还還 运運 远遠 选選 边邊 导導 '
         '将將 岁歲 热熱 图圖 团團 园園 圆圓 广廣 庆慶 张張 强強 归歸 当當 录錄 彻徹 忆憶 态態 怀懷 恶惡 悬懸 惊驚 愿願 戏戲 户戶').split()
SIMPLIFIED, TRADITIONAL = {p[0] for p in PAIRS}, {p[1] for p in PAIRS}


def in_script(text, kind):
    """True when most of the document's letters are in the script `kind` (FineWeb-2's language labels are not perfect)."""
    letters = [c for c in text[:5000] if c.isalpha()]
    if not letters:
        return False
    if kind == 'hangul':
        return sum('\uac00' <= c <= '\ud7a3' for c in letters) >= 0.6 * len(letters)
    kana = sum('\u3040' <= c <= '\u30ff' for c in letters)
    han = sum('\u4e00' <= c <= '\u9fff' for c in letters)
    if kind == 'kana':
        return kana >= 20 and kana + han >= 0.6 * len(letters)
    simplified, traditional = sum(c in SIMPLIFIED for c in text[:5000]), sum(c in TRADITIONAL for c in text[:5000])
    wanted = simplified if kind == 'simplified' else traditional
    return (kana == 0 and han >= 0.6 * len(letters) and simplified + traditional >= 3
            and wanted >= 4 * (simplified + traditional - wanted))


def encode_documents(texts, kind=None):
    """[BOS] text [EOS] for every document (in the script `kind`, when given), as kelus's raw corpora store them."""
    if len(_banned):
        texts = [text for text in texts if text and not contaminated(text, _banned)]
    if not isinstance(_worker, Tokenizer):
        if kind is not None:
            texts = [text for text in texts if text and in_script(text, kind)]
        return np.frombuffer(_worker.encode_documents(texts), dtype='<u2')
    out = []
    for text in texts:
        if text and (kind is None or in_script(text, kind)):
            out.append(BOS)
            out.extend(_worker.encode(text))
            out.append(EOS)
    return np.asarray(out, dtype='<u2')




def find_file(settings, name):
    """run_dir/<name>, else the same file in an attached earlier output of this notebook (or a dataset). With several
    attached, the one beside the progress.json that has seen the most tokens: the latest session's."""
    run = Path(settings['run_dir'])
    if (run / name).exists():
        return run / name
    root = Path(settings['input_root'])
    # In a folder named like run_dir (a notebook's output), or loose at a dataset's top level (files uploaded as they are).
    patterns = [f'{"*/" * depth}{run.name}/{name}' for depth in (1, 2, 3)] + [f'{"*/" * depth}{name}' for depth in (1, 2, 3)]
    found = list(dict.fromkeys(path for pattern in patterns for path in (sorted(root.glob(pattern)) if root.is_dir() else [])))

    def tokens(path):
        try:
            return json.loads((path.parent / 'progress.json').read_text())['tokens_seen']
        except (OSError, ValueError, KeyError):
            return -1
    return max(found, key=tokens) if found else None


def read_progress(settings):
    path = find_file(settings, 'progress.json')
    return json.loads(path.read_text()) if path else {'session': 0, 'tokens_seen': 0, 'step': 0}


def source_files(source):
    """Sorted data files of one source: Hugging Face parquet files, or local ones ('local' glob) for testing."""
    if 'local' in source:
        return sorted(glob.glob(source['local']))
    from huggingface_hub import HfApi
    files = HfApi().list_repo_files(source['repo'], repo_type='dataset', revision=source.get('revision'))
    return sorted(f for f in files if f.startswith(source['prefix']) and f.endswith('.parquet'))


def fetch(source, name, folder):
    if 'local' in source:
        return Path(name)
    from huggingface_hub import hf_hub_download

    return Path(hf_hub_download(source['repo'], name, repo_type='dataset', revision=source.get('revision'), local_dir=folder))


def parquet_texts(path, documents, column='text'):
    """Lists of up to `documents` texts from a parquet file's text column, read lazily."""
    import pyarrow.parquet as pq
    for batch in pq.ParquetFile(path).iter_batches(columns=[column], batch_size=documents):
        yield batch.column(column).to_pylist()


def multilingual(settings):
    return any('share' in source for source in settings['sources'])


def source_key(source):
    return source.get('local') or f"{source['repo']}/{source.get('prefix', '')}"


def start_file(source, session, cursors):
    """The number of the first file of `source` that a session reads: the one after the last file read before (progress
    'next_file', which both this script and kani_tpu.py keep), else session x per_session + offset. Numbers wrap around
    the source's files."""
    key = source_key(source)
    return cursors[key] if key in cursors else session * source['per_session'] + source.get('offset', 0)


def data_files(settings):
    """Each source's training files, and the held-out validation file(s).

    English runs hold out the first source's last file. Multilingual runs hold out a file of every source (its
    `validation_file`, else its last file) and can start a source `offset` files in, so two sources reading the
    same files (Simplified and Traditional Chinese) read different ones.
    """
    lists, validation = [], [] if multilingual(settings) else None
    for index, source in enumerate(settings['sources']):
        files = source_files(source)
        if not multilingual(settings):
            if index == 0:
                validation, files = files[-1], files[:-1]
        else:
            held = source.get('validation_file') or files[-1]
            validation.append((index, held))
            files = [name for name in files if name != held]
        lists.append(files)
    return lists, validation


def session_plan(settings, session, cursors=None):
    """(source index, file) pairs to tokenize this session, the held-out validation file(s), and where the next
    session starts in each source."""
    lists, validation = data_files(settings)
    plan, after = [], {}
    for index, (source, files) in enumerate(zip(settings['sources'], lists)):
        start = start_file(source, session, cursors or {})
        plan += [(index, files[(start + j) % len(files)]) for j in range(source['per_session'])]
        after[source_key(source)] = start + source['per_session']
    return plan, validation, after


def validation_tokens(settings, validation, pool, folder):
    """The held-out documents' tokens: validation_documents in all, shared between the held-out files."""
    held_out = validation if multilingual(settings) else [(0, validation)]
    parts = []
    for index, name in held_out:
        source = settings['sources'][index]
        path = fetch(source, name, folder)
        encode = functools.partial(encode_documents, kind=source.get('filter'))
        wanted, texts = settings['validation_documents'] // len(held_out), []
        for batch in parquet_texts(path, 2000, source.get('column', 'text')):
            texts += [text for text in batch if source.get('filter') is None or in_script(text, source['filter'])]
            if len(texts) >= wanted:
                break
        texts = texts[:wanted]
        parts += pool.map(encode, [texts[i:i + 250] for i in range(0, len(texts), 250)])
        if 'local' not in source:
            path.unlink(missing_ok=True)
    return np.concatenate(parts)


def tokenize_file(source, name, out, limit, pool, workers, folder):
    """Append one parquet file's documents to `out` as tokens; returns the tokens written (about `limit` at most)."""
    path = fetch(source, name, folder)
    reader, written = parquet_texts(path, 500, source.get('column', 'text')), 0
    encode = functools.partial(encode_documents, kind=source.get('filter'))
    with open(out, 'ab') as stream:

        while written < limit and (group := list(itertools.islice(reader, 4 * workers))):
            for tokens in pool.map(encode, group):
                tokens.tofile(stream)
                written += len(tokens)
    if 'local' not in source:
        path.unlink(missing_ok=True)
    return written


def attached(settings):
    """The folders under input_root, two levels deep, to show what Kaggle mounted."""
    root = Path(settings['input_root'])
    if not root.is_dir():
        return [f'{root} does not exist']
    return sorted(str(p.relative_to(root)) for p in root.glob('*/*')) or sorted(p.name for p in root.iterdir()) or ['(nothing attached)']


def prepare(settings):
    from multiprocessing import Pool
    checkpoint = find_file(settings, 'checkpoint.pt')
    print('attached inputs:', ', '.join(attached(settings)[:20]), flush=True)
    Tokenizer.load(settings['tokenizer'])
    print('tokenizing with', 'Rust (kani_tokenize)' if fast_tokenizer(settings['tokenizer']) else 'Python', flush=True)
    if checkpoint:
        print(f'continuing from {checkpoint}', flush=True)
    elif settings['continue_from_previous']:
        raise SystemExit('No earlier checkpoint.pt found, so this session would start over from zero. '
                         'Attach this notebook\'s latest output (Add Input → Your Work → Notebooks), '
                         'or set continue_from_previous to False to begin a new run.')
    else:
        print('no earlier checkpoint: starting a NEW run from zero', flush=True)
    banned = find_file(settings, 'contamination.npy')
    if banned:
        print(f'dropping documents that share 13 words with the test sets ({len(np.load(banned)):,} runs, {banned})', flush=True)
        run = Path(settings['run_dir'])
        if banned.parent != run:  # it travels on with this session's output
            run.mkdir(parents=True, exist_ok=True)
            banned = Path(shutil.copy(banned, run / 'contamination.npy'))
    elif settings['decontaminate']:
        raise SystemExit('decontaminate is on but no contamination.npy was found: attach the dataset that holds it '
                         '(made by "kani_advanced.py grams") beside the checkpoint.')
    progress = read_progress(settings)
    session, out = progress['session'], Path(settings['data_dir'])
    plan, validation, after = session_plan(settings, session, progress.get('next_file'))
    key = {'session': session, 'plan': plan, 'validation': validation, 'tokenizer': settings['tokenizer'],
           'session_tokens': settings['session_tokens'], 'validation_documents': settings['validation_documents'],
           'decontaminated': bool(banned)}
    key = json.loads(json.dumps(key))
    if (out / 'manifest.json').exists() and json.loads((out / 'manifest.json').read_text()).get('key') == key:
        print(f'session {session}: {out} is already prepared')
        return
    out.mkdir(parents=True, exist_ok=True)
    for name in ('manifest.json', 'train.bin', 'validation.bin'):
        (out / name).unlink(missing_ok=True)
    folder, started, workers = str(out / 'downloads'), time.time(), settings['workers'] or os.cpu_count()
    print(f'session {session}: {", ".join(name for _, name in plan)}', flush=True)
    with Pool(workers, initializer=_start_worker, initargs=(settings['tokenizer'], banned and str(banned))) as pool:
        validation_tokens(settings, validation, pool, folder).tofile(out / 'validation.bin')
        total = 0
        for index, name in plan:
            source = settings['sources'][index]

            share = (int(settings['session_tokens'] * source['share'] / source['per_session']) if multilingual(settings)
                     else settings['session_tokens'] // len(plan))
            written = tokenize_file(source, name, out / 'train.bin', share, pool, workers, folder)
            total += written
            print(f'  {name}: {written:,} tokens ({time.time() - started:.0f}s)', flush=True)
    validation_count = (out / 'validation.bin').stat().st_size // 2
    (out / 'manifest.json').write_text(json.dumps({'key': key, 'train_tokens': total, 'validation_tokens': validation_count,
                                                   'next_file': after}, indent=1))
    print(f'{total:,} training tokens and {validation_count:,} validation tokens in {time.time() - started:.0f}s')




def rope_tables(context, head_width):
    """kelus.model's rotary tables: angle = position * 10000^(-i/half), repeated for both halves."""
    half = head_width // 2
    angles = np.arange(context)[:, None] * 10000.0 ** (-np.arange(half) / half)
    cos = np.concatenate([np.cos(angles)] * 2, axis=1).astype(np.float32)
    sin = np.concatenate([np.sin(angles)] * 2, axis=1).astype(np.float32)
    return torch.from_numpy(cos), torch.from_numpy(sin)


def rotate(x, cos, sin):
    half = x.shape[-1] // 2
    return x * cos + torch.cat([-x[..., half:], x[..., :half]], dim=-1) * sin


class Block(nn.Module):
    def __init__(self, width, heads):
        super().__init__()
        self.heads = heads
        self.norm1, self.norm2 = nn.LayerNorm(width, eps=1e-5), nn.LayerNorm(width, eps=1e-5)
        self.q, self.k, self.v, self.projection = (nn.Linear(width, width, bias=False) for _ in range(4))
        self.up, self.down = nn.Linear(width, 4 * width, bias=False), nn.Linear(4 * width, width, bias=False)

    def forward(self, x, cos, sin):
        b, t, d = x.shape
        h = self.norm1(x)
        q, k, v = (layer(h).view(b, t, self.heads, d // self.heads).transpose(1, 2) for layer in (self.q, self.k, self.v))
        q, k = rotate(q, cos[:t], sin[:t]).to(v.dtype), rotate(k, cos[:t], sin[:t]).to(v.dtype)
        mixed = F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(b, t, d)
        x = x + self.projection(mixed)
        return x + self.down(F.gelu(self.up(self.norm2(x)), approximate='tanh'))


class Kani(nn.Module):
    def __init__(self, vocab, width, layers, heads, context, checkpointing=False):
        super().__init__()
        self.context, self.checkpointing = context, checkpointing
        self.token = nn.Embedding(vocab, width)
        self.blocks = nn.ModuleList(Block(width, heads) for _ in range(layers))
        self.final = nn.LayerNorm(width, eps=1e-5)
        cos, sin = rope_tables(context, width // heads)
        self.register_buffer('cos', cos, persistent=False)
        self.register_buffer('sin', sin, persistent=False)
        for name, p in self.named_parameters():
            if p.dim() == 2:
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * layers) if name.endswith(('projection.weight', 'down.weight')) else 0.02)

    def forward(self, ids):
        x = self.token(ids)
        for block in self.blocks:
            if self.checkpointing and self.training:
                x = torch.utils.checkpoint.checkpoint(block, x, self.cos, self.sin, use_reentrant=False)
            else:
                x = block(x, self.cos, self.sin)
        return F.linear(self.final(x), self.token.weight)


def build(settings, vocab):
    return Kani(vocab, settings['width'], settings['layers'], settings['heads'], settings['context'], settings['gradient_checkpointing'])


def kani_parameters(net):
    """The model's weights under kelus.model's names and layouts (matrices are input x output)."""
    state = {k: v.detach().float().cpu().numpy() for k, v in net.state_dict().items()}
    values = {'token': state['token.weight'], 'final.gain': state['final.weight'], 'final.bias': state['final.bias']}
    for i in range(len(net.blocks)):
        p = f'blocks.{i}.'
        for norm in ('norm1', 'norm2'):
            values[f'block{i}.{norm}.gain'] = state[p + norm + '.weight']
            values[f'block{i}.{norm}.bias'] = state[p + norm + '.bias']
        for matrix in ('q', 'k', 'v', 'projection', 'up', 'down'):
            values[f'block{i}.{matrix}'] = np.ascontiguousarray(state[p + matrix + '.weight'].T)
    return values


def export(net, settings, tokenizer, progress, path):
    """A checkpoint in kelus's format ("kelus-b1" version 1), loadable by kelus.model.Model.load."""
    config = {'context': settings['context'], 'width': settings['width'], 'heads': settings['heads'],
              'layers': settings['layers'], 'seed': settings['seed'], 'vocab_size': tokenizer.vocab_size,
              'positions': 'rope', 'tie_embeddings': True, 'dropout': 0.0}
    metadata = {'display_name': 'Kani Advanced', 'steps': progress['step'], 'tokens_seen': progress['tokens_seen'],
                'sessions': progress['session'], 'stage': 'pretraining (raw text); needs instruction tuning before chat',
                'data': [s.get('repo', s.get('local')) for s in settings['sources']], 'trained_on': 'Kaggle (kaggle/kani_advanced.py)'}
    header = {'format': 'kelus-b1', 'version': 1, 'name': 'Kani Seal 3', 'config': config,
              'tokenizer': tokenizer.to_dict(), 'metadata': metadata}
    values = {'parameter.' + k: v for k, v in kani_parameters(net).items()}
    temp = Path(str(path) + '.tmp.npz')
    np.savez(temp, header=np.array(json.dumps(header)), **values)
    temp.replace(path)
    print(f'exported {path} ({sum(p.numel() for p in net.parameters()):,} parameters, '
          f'{progress["tokens_seen"] / 1e9:.2f}B tokens seen)', flush=True)


@torch.no_grad()
def generate(net, tokenizer, prompt, tokens=60, temperature=0.8, top_k=40, seed=0):
    device = next(net.parameters()).device
    ids = [BOS] + tokenizer.encode(prompt)
    generator = torch.Generator(device=device).manual_seed(seed)
    for _ in range(tokens):
        with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
            logits = net(torch.tensor([ids[-net.context:]], device=device))[0, -1].float()
        logits[[PAD, BOS, USER, ASSISTANT]] = -float('inf')
        values, indices = logits.topk(top_k)
        token = int(indices[torch.multinomial((values / temperature).softmax(-1), 1, generator=generator)])
        if token == EOS:
            break
        ids.append(token)
    return tokenizer.decode(ids[1:])


PROMPTS = ['The capital of France is', 'Photosynthesis is the process by which', 'In 1969, astronauts',
           'To sort a list in Python, you can', 'The main causes of the First World War were']

PROMPTS_MULTILINGUAL = PROMPTS[:3] + ['Der Himmel ist blau, weil', 'La capitale de la France est', 'El agua hierve a',
                                      'O Brasil é o maior país', '한국의 수도는', '日本で一番高い山は', '中国的首都是', '台灣最高的山是']




def learning_rate(settings, tokens, rewarm_from=None):
    """Warmup, then constant, then a linear decay over the last `decay_fraction` of target_tokens. After a retokenize
    (a fresh optimizer from `rewarm_from` tokens) the rate ramps up again over `rewarm_tokens`."""
    rate = schedule(settings, tokens)
    if rewarm_from is not None and tokens - rewarm_from < settings['rewarm_tokens']:
        rate *= (tokens - rewarm_from + settings['batch_tokens']) / settings['rewarm_tokens']
    return rate


def schedule(settings, tokens):
    peak, target = settings['lr'], settings['target_tokens']
    if tokens < settings['warmup_tokens']:
        return peak * (tokens + settings['batch_tokens']) / settings['warmup_tokens']
    decay_start = target * (1 - settings['decay_fraction'])
    if tokens < decay_start:
        return peak
    done = min(1.0, (tokens - decay_start) / max(target - decay_start, 1))
    return peak * (1 - done * (1 - settings['min_lr_ratio']))


def windows(stream, count, context, rng):
    starts = rng.integers(0, len(stream) - context - 1, size=count)
    blocks = torch.from_numpy(np.stack([stream[s:s + context + 1] for s in starts]).astype(np.int64))
    return blocks[:, :-1], blocks[:, 1:]


class Blocks:
    """Training text in a shuffled order without repeats: every context-long block of the session's text is read once
    before any block comes back (random windows re-read about a third of the text and never reach another third).
    The position follows the steps taken this session, so a restarted run continues where it stopped."""

    def __init__(self, stream, context, micro_batch, seed, session, rank, world, done_micro_steps):
        self.stream, self.context, self.micro_batch = stream, context, micro_batch
        self.count = (len(stream) - 1) // context
        self.seed, self.session, self.rank, self.world = seed, session, rank, world
        self.position = done_micro_steps * micro_batch * world
        self.order, self.epoch = None, -1

    def block(self, index):
        epoch, within = divmod(index, self.count)
        if epoch != self.epoch:
            self.order, self.epoch = np.random.default_rng([self.seed, self.session, epoch]).permutation(self.count), epoch
        start = int(self.order[within]) * self.context
        return self.stream[start:start + self.context + 1]

    def next(self):
        first = self.position + self.rank * self.micro_batch
        self.position += self.world * self.micro_batch
        blocks = torch.from_numpy(np.stack([self.block(first + j) for j in range(self.micro_batch)]).astype(np.int64))
        return blocks[:, :-1], blocks[:, 1:]


@torch.no_grad()
def validate(net, stream, settings, device):
    net.eval()
    rng, total = np.random.default_rng(0), 0.0
    for _ in range(settings['eval_batches']):
        x, y = windows(stream, settings['micro_batch'], settings['context'], rng)
        with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
            logits = net(x.to(device))
        total += F.cross_entropy(logits.float().flatten(0, 1), y.to(device).flatten()).item()
    net.train()
    return total / settings['eval_batches']


def save_checkpoint(settings, net, opt, scaler, progress):
    run = Path(settings['run_dir'])
    temp = run / 'checkpoint.tmp'
    torch.save({'net': net.state_dict(), 'opt': opt and opt.state_dict(), 'scaler': scaler.state_dict(), 'progress': progress,
                'settings': settings}, temp)
    temp.replace(run / 'checkpoint.pt')
    (run / 'progress.json').write_text(json.dumps(progress, indent=1))


def train(settings):
    distributed = int(os.environ.get('WORLD_SIZE', '1')) > 1
    cuda = torch.cuda.is_available()
    if distributed:
        torch.distributed.init_process_group('nccl' if cuda else 'gloo')
    rank = torch.distributed.get_rank() if distributed else 0
    world = torch.distributed.get_world_size() if distributed else 1
    local = int(os.environ.get('LOCAL_RANK', '0'))
    device = torch.device('cuda', local) if cuda else torch.device('cpu')
    if cuda:
        torch.cuda.set_device(device)
        torch.backends.cuda.matmul.allow_tf32 = True
    main = rank == 0
    per_step = world * settings['micro_batch'] * settings['context']
    if settings['batch_tokens'] % per_step:
        raise SystemExit(f"batch_tokens must be a multiple of GPUs x micro_batch x context = {per_step}")
    accumulate = settings['batch_tokens'] // per_step

    tokenizer = Tokenizer.load(settings['tokenizer'])
    data = Path(settings['data_dir'])
    if not (data / 'manifest.json').exists():
        raise SystemExit(f'No prepared text in {data}: run "prepare" first (it may have been interrupted)')
    manifest = json.loads((data / 'manifest.json').read_text())
    prepared_session = manifest['key']['session']
    stream = np.memmap(data / 'train.bin', dtype='<u2', mode='r')
    held_out = np.memmap(data / 'validation.bin', dtype='<u2', mode='r')
    run = Path(settings['run_dir'])
    run.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(settings['seed'])
    net = build(settings, tokenizer.vocab_size).to(device)
    decay = [p for n, p in net.named_parameters() if p.dim() == 2]
    no_decay = [p for n, p in net.named_parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([{'params': decay, 'weight_decay': settings['weight_decay']}, {'params': no_decay, 'weight_decay': 0.0}],
                            lr=settings['lr'], betas=(0.9, 0.95), eps=1e-8, fused=cuda)
    scaler = torch.amp.GradScaler('cuda', enabled=cuda)
    progress = {'session': 0, 'tokens_seen': 0, 'step': 0, 'history': []}
    checkpoint = find_file(settings, 'checkpoint.pt')
    if checkpoint:
        state = torch.load(checkpoint, map_location='cpu', weights_only=True)
        for key in ('width', 'layers', 'heads', 'context'):
            if state['settings'][key] != settings[key]:
                raise SystemExit(f'{checkpoint} has {key}={state["settings"][key]}; keep the model settings or use a new run_dir')
        if state['net']['token.weight'].shape[0] != tokenizer.vocab_size:
            raise SystemExit(f'{checkpoint} has a {state["net"]["token.weight"].shape[0]:,}-token vocabulary, the tokenizer '
                             f'{tokenizer.vocab_size:,}: run retokenize first')
        net.load_state_dict(state['net'])
        if state['opt'] is not None:
            opt.load_state_dict(state['opt'])
            scaler.load_state_dict(state['scaler'])
        progress = state['progress']
        if main:
            print(f'resumed from {checkpoint}: session {progress["session"]}, step {progress["step"]:,}, '
                  f'{progress["tokens_seen"] / 1e9:.2f}B tokens', flush=True)
    model = nn.parallel.DistributedDataParallel(net, device_ids=[local] if cuda else None) if distributed else net
    forward = torch.compile(model) if settings['compile'] else model
    if main and prepared_session != progress['session']:
        print(f'! the prepared text is for session {prepared_session}, not {progress["session"]}: run prepare first '
              'for fresh text (training continues on the old text)', flush=True)
    if main:
        print(json.dumps({'parameters': sum(p.numel() for p in net.parameters()), 'gpus': world, 'device': str(device),
                          'session': progress['session'], 'train_tokens_available': len(stream),
                          'tokens_per_step': settings['batch_tokens'], 'accumulate': accumulate}), flush=True)


    session_start = progress['history'][-1]['step'] if progress.get('history') else 0
    blocks = Blocks(stream, settings['context'], settings['micro_batch'], settings['seed'], progress['session'], rank, world,
                    max(0, progress['step'] - session_start) * accumulate)
    started = last_checkpoint = time.time()
    budget, stop, stopping = settings['hours'] * 3600, torch.zeros(1, device=device), False
    total, count, window = torch.zeros((), device=device), 0, time.time()
    validation_loss = None
    while not stopping:
        lr = learning_rate(settings, progress['tokens_seen'], progress.get('rewarm_from'))
        for group in opt.param_groups:
            group['lr'] = lr
        for micro in range(accumulate):
            x, y = blocks.next()
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            sync = nullcontext() if not distributed or micro == accumulate - 1 else model.no_sync()
            with sync:
                with torch.autocast(device.type, dtype=torch.float16, enabled=cuda):
                    logits = forward(x)
                loss = F.cross_entropy(logits.float().flatten(0, 1), y.flatten()) / accumulate
                scaler.scale(loss).backward()
            total += loss.detach()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        progress['step'] += 1
        progress['tokens_seen'] += settings['batch_tokens']
        count += 1

        if progress['step'] % 10 == 0:
            done = time.time() - started > budget or progress['tokens_seen'] >= settings['target_tokens']
            stop.fill_(float(done))
            if distributed:
                torch.distributed.broadcast(stop, 0)
            stopping = bool(stop.item())
        if main and (progress['step'] % settings['eval_every'] == 0 or stopping):
            validation_loss = validate(net, held_out, settings, device)
        if main and (progress['step'] % 20 == 0 or stopping):
            rate = count * settings['batch_tokens'] / (time.time() - window)
            status = {'step': progress['step'], 'tokens': f"{progress['tokens_seen'] / 1e9:.3f}B", 'loss': round(total.item() / count, 4),
                      'validation_loss': validation_loss and round(validation_loss, 4), 'lr': f'{lr:.2e}', 'tokens_per_second': round(rate),
                      'hours_left': round(max(0.0, budget - (time.time() - started)) / 3600, 2)}
            if progress['step'] == 20 and cuda:
                status['gpu_memory_gb'] = round(torch.cuda.max_memory_allocated() / 2 ** 30, 1)
            print(json.dumps(status), flush=True)
            (run / 'status.json').write_text(json.dumps({**status, 'state': 'running', 'updated': time.time()}))
            total, count, window = torch.zeros((), device=device), 0, time.time()
        if main and time.time() - last_checkpoint > settings['checkpoint_minutes'] * 60:
            save_checkpoint(settings, net, opt, scaler, progress)
            last_checkpoint = time.time()
            print(f'checkpoint at step {progress["step"]:,}', flush=True)

    if main:
        progress['history'].append({'session': progress['session'], 'step': progress['step'], 'tokens_seen': progress['tokens_seen'],
                                    'validation_loss': validation_loss, 'hours': round((time.time() - started) / 3600, 2)})
        if prepared_session == progress['session'] and 'next_file' in manifest:
            progress['next_file'] = manifest['next_file']
        progress['session'] += 1
        save_checkpoint(settings, net, opt, scaler, progress)
        export(net, settings, tokenizer, progress, run / 'kani-advanced.npz')
        complete = progress['tokens_seen'] >= settings['target_tokens']
        (run / 'status.json').write_text(json.dumps({'step': progress['step'], 'tokens_seen': progress['tokens_seen'],
                                                     'validation_loss': validation_loss, 'state': 'complete' if complete else 'stopped'}))
        net.eval()
        prompts = PROMPTS_MULTILINGUAL if tokenizer.version == 3 else PROMPTS
        samples = {prompt: generate(net, tokenizer, prompt) for prompt in prompts}
        (run / 'samples.json').write_text(json.dumps(samples, indent=1, ensure_ascii=False))
        for prompt, text in samples.items():
            print(f'--- {text}', flush=True)
        print(('target reached: ' if complete else 'session done: ') + f'{progress["tokens_seen"] / 1e9:.2f}B of '
              f'{settings["target_tokens"] / 1e9:.1f}B tokens, validation loss {validation_loss:.4f}', flush=True)
    if distributed:
        torch.distributed.destroy_process_group()


def retokenize(settings, checkpoint, old_path):
    """Move `checkpoint` (trained with the tokenizer at `old_path`) to settings['tokenizer'], as this run's start.

    The blocks keep everything they learned. Each new token's embedding (also the output layer, which is tied)
    starts as the mean of the old embeddings of the old tokens that spell it, so known words keep their meaning.
    The optimizer starts afresh, so training warms the learning rate up again (rewarm_tokens).
    """
    state = torch.load(checkpoint, map_location='cpu', weights_only=True)
    old, new = Tokenizer.load(old_path), Tokenizer.load(settings['tokenizer'])
    table = state['net']['token.weight']
    if table.shape[0] != old.vocab_size:
        raise SystemExit(f'{checkpoint} does not use {old_path}')
    rows = []
    for token in range(new.vocab_size):
        if token < 261:
            rows.append(table[token])
            continue
        piece = new.pieces[token]
        try:
            ids = old.encode(piece.decode('utf-8'))
        except UnicodeDecodeError:
            ids = list(piece)
        rows.append(table[ids].mean(0))
    state['net']['token.weight'] = torch.stack(rows)
    progress = {**state['progress'], 'rewarm_from': state['progress']['tokens_seen'], 'retokenized_from': str(checkpoint)}
    run = Path(settings['run_dir'])
    run.mkdir(parents=True, exist_ok=True)
    torch.save({'net': state['net'], 'opt': None, 'scaler': None, 'progress': progress,
                'settings': {**state['settings'], 'tokenizer': settings['tokenizer']}}, run / 'checkpoint.tmp')
    (run / 'checkpoint.tmp').replace(run / 'checkpoint.pt')
    (run / 'progress.json').write_text(json.dumps(progress, indent=1))
    print(f'retokenized {checkpoint} ({old.vocab_size:,} tokens) to {new.vocab_size:,} tokens in {run}: step {progress["step"]:,}, '
          f'{progress["tokens_seen"] / 1e9:.2f}B tokens', flush=True)




def grow(settings, base, progress):
    """Start a deeper run from a trained Kani (`base`, kelus's npz format): its blocks in order, with a copy of every
    k-th block inserted after it until there are settings['layers'] blocks. A copy starts with its attention projection
    and feed-forward `down` at zero, so it adds nothing and the grown model is exactly the base (LLaMA Pro's block
    expansion); training then grows the copies into new layers. The optimizer starts fresh: train() ramps the
    learning rate up again over rewarm_tokens from `progress['rewarm_from']`."""
    with np.load(base) as z:
        header, values = json.loads(str(z['header'])), {k.removeprefix('parameter.'): z[k] for k in z.files if k != 'header'}
    config, layers = header['config'], settings['layers']
    for key in ('width', 'heads', 'context'):
        if config[key] != settings[key]:
            raise SystemExit(f'{base} has {key}={config[key]}, the settings {settings[key]}')
    extra, old = layers - config['layers'], config['layers']
    if not 0 < extra <= old or old % extra:
        raise SystemExit(f'grow adds 1 to {old} blocks, a whole fraction of {old}; asked for {extra}')
    every, order = old // extra, []
    for i in range(old):
        order.append((i, False))
        if i % every == every - 1:
            order.append((i, True))
    state = {'token.weight': values['token'], 'final.weight': values['final.gain'], 'final.bias': values['final.bias']}
    for j, (i, copy) in enumerate(order):
        for norm in ('norm1', 'norm2'):
            state[f'blocks.{j}.{norm}.weight'] = values[f'block{i}.{norm}.gain']
            state[f'blocks.{j}.{norm}.bias'] = values[f'block{i}.{norm}.bias']
        for matrix in ('q', 'k', 'v', 'projection', 'up', 'down'):
            weight = np.ascontiguousarray(values[f'block{i}.{matrix}'].T)
            state[f'blocks.{j}.{matrix}.weight'] = np.zeros_like(weight) if copy and matrix in ('projection', 'down') else weight
    net = build(settings, values['token'].shape[0])
    net.load_state_dict({k: torch.from_numpy(np.array(v, dtype=np.float32)) for k, v in state.items()})
    progress = {**progress, 'rewarm_from': progress['tokens_seen'],
                'grown_from': {'checkpoint': str(base), 'layers': old, 'tokens_seen': progress['tokens_seen'], 'copied_blocks': [i for i, c in order if c]}}
    run = Path(settings['run_dir'])
    run.mkdir(parents=True, exist_ok=True)
    save_checkpoint(settings, net, None, torch.amp.GradScaler('cuda', enabled=False), progress)
    print(f'grew {base} from {old} to {layers} blocks ({sum(p.numel() for p in net.parameters()):,} parameters); '
          f'copies after blocks {[i for i, c in order if c]}', flush=True)
    return net


def load_trained(settings):
    checkpoint = find_file(settings, 'checkpoint.pt')
    if checkpoint is None:
        raise SystemExit('No checkpoint.pt yet; train first.')
    state = torch.load(checkpoint, map_location='cpu', weights_only=True)
    tokenizer = Tokenizer.load(settings['tokenizer'])
    net = build({**settings, **{k: state['settings'][k] for k in ('width', 'layers', 'heads', 'context')}}, tokenizer.vocab_size)
    net.load_state_dict(state['net'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    return net.to(device).eval(), tokenizer, state


def main():
    import sys
    if 'ipykernel' in sys.modules:
        print('This is the training script, not the notebook. On Kaggle use File → Import Notebook with '
              'kaggle/kani-advanced.ipynb, which writes this file and runs it.')
        return
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument('command', choices=['prepare', 'train', 'sample', 'export', 'retokenize', 'grams', 'grow'])
    parser.add_argument('--settings', default='settings.json')
    parser.add_argument('--prompt', action='append', help='sample: text to continue (repeatable)')
    parser.add_argument('--tokens', type=int, default=80)
    parser.add_argument('--checkpoint', help='retokenize: the checkpoint.pt to move to the new vocabulary')
    parser.add_argument('--old-tokenizer', help="retokenize: that checkpoint's tokenizer")
    parser.add_argument('--tests', help="grams: the folder of kelus.standard_eval's test files")
    parser.add_argument('--out', help='grams: where to write contamination.npy')
    parser.add_argument('--base', help='grow: the trained kani-advanced.npz to grow')
    parser.add_argument('--progress', help="grow: the base's progress.json (session, step, tokens_seen, next_file)")
    args = parser.parse_args()
    if args.command == 'grams':
        if not (args.tests and args.out):
            raise SystemExit('grams needs --tests and --out')
        write_grams(args.tests, args.out)
        return
    settings = load_settings(args.settings)
    if args.command == 'prepare':
        prepare(settings)
    elif args.command == 'train':
        train(settings)
    elif args.command == 'grow':
        if not (args.base and args.progress):
            raise SystemExit('grow needs --base and --progress')
        grow(settings, args.base, json.loads(Path(args.progress).read_text()))
    elif args.command == 'retokenize':
        if not (args.checkpoint and args.old_tokenizer):
            raise SystemExit('retokenize needs --checkpoint and --old-tokenizer')
        retokenize(settings, args.checkpoint, args.old_tokenizer)
    elif args.command == 'export':
        net, tokenizer, state = load_trained(settings)
        Path(settings['run_dir']).mkdir(parents=True, exist_ok=True)
        export(net, {**settings, **state['settings'], 'run_dir': settings['run_dir']}, tokenizer, state['progress'],
               Path(settings['run_dir']) / 'kani-advanced.npz')
    else:
        net, tokenizer, _ = load_trained(settings)
        for i, prompt in enumerate(args.prompt or (PROMPTS_MULTILINGUAL if tokenizer.version == 3 else PROMPTS)):
            print(f'--- {generate(net, tokenizer, prompt, tokens=args.tokens, seed=i)}')


if __name__ == '__main__':
    main()

## This session's text
Downloads one new FineWeb-Edu file and one Cosmopedia file and turns them into Kani's tokens (about 10 minutes).

In [ ]:
!python kani_advanced.py prepare --settings settings.json
# A failed prepare (for example no earlier checkpoint attached) stops the notebook here, not with errors further down.
import kani_advanced
from pathlib import Path
if not (Path(kani_advanced.load_settings('settings.json')['data_dir']) / 'manifest.json').exists():
    raise RuntimeError('Stopped before training: read the message just above.')

## Train

A log line every 20 steps. `loss` and `validation_loss` should fall over the session and across sessions.
The first line shows `gpu_memory_gb`; if a run stops with "CUDA out of memory", set `micro_batch` to 2.

In [ ]:
gpu_count = max(1, torch.cuda.device_count())
!torchrun --nnodes=1 --nproc_per_node={gpu_count} --master_addr=127.0.0.1 --master_port=29500 kani_advanced.py train --settings settings.json

## How it writes now

In [ ]:
progress = json.load(open('kani-advanced/progress.json'))
for session in progress['history']:
    print(session)
for prompt, text in json.load(open('kani-advanced/samples.json')).items():
    print('\n---', text)

## Bring it to Kani

Download `kani-advanced/kani-advanced.npz` from the Output tab and put it in `KelusModelB1/checkpoints/`.
It is a raw-text model: it continues text but does not chat yet. Chat tuning (Kani's conversations, web
search and reading data) and the capability tests then run on the Mac, before it replaces the current Kani.